# Train the point cloud denoiser (Kaggle GPU)

CPU training is not viable for this model: measured at 3.3 s per batch of 8
patches, one epoch over 40 shapes is about 4.6 hours and a 50-epoch run is
around 10 days. A T4 turns that into hours.

## Before running

1. **Add your code** - *Add Input* -> *Upload* -> the `pointdenoise-code.zip`
   produced by `scripts/package_for_kaggle.py`.
2. **Add the data** - *Add Input* -> *Upload* -> `pointdenoise-data.zip`
   (the unpacked ScoreDenoise archive, see docs/benchmark.md).
3. **GPU on** - *Settings* -> *Accelerator* -> **GPU T4 x2**.
4. Run all.

## After running

Download `best.pt` and `benchmark.txt` from the output panel. Put the
checkpoint in `runs/kaggle/` locally and the results table in `results/`.


In [ ]:
import glob, os, subprocess, sys

# Locate the uploaded code and data at whatever depth Kaggle unpacked them to.
def find_dir(marker, root="/kaggle/input"):
    for base, dirs, files in os.walk(root):
        if marker in dirs or marker in files:
            return base
    return None

CODE = find_dir("pointdenoise")
DATA = find_dir("examples")
print("code:", CODE)
print("data:", DATA)
assert CODE, f"pointdenoise package not found. /kaggle/input holds: {os.listdir('/kaggle/input')}"
assert DATA, "benchmark data not found (looking for an 'examples' directory)"

sys.path.insert(0, CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "trimesh", "rtree"], check=False)

import torch
print("\ntorch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - enable it in Settings")


In [ ]:
from pointdenoise.benchmark import load_released_set, load_training_clouds
from pointdenoise.data import Shape
import numpy as np

train_clouds = load_training_clouds(DATA, "PUNet", "sparse")
print(f"{len(train_clouds)} training shapes")

# Noise is applied fresh each epoch by the dataset, so the level here only
# seeds the initial corruption; see NOISE_LEVEL below.
NOISE_LEVEL = 0.02
rng = np.random.default_rng(0)
shapes = [Shape(pts, noise_level=NOISE_LEVEL, rng=rng) for _, pts in train_clouds]
print(f"points per shape: {shapes[0].clean.shape[0]}")


## Sanity check before spending GPU hours

Confirms the harness reproduces a published number on this data. If this does
not say PASS, stop: the numbers the run produces would not be comparable to
anything and there is no point training first.


In [ ]:
from pointdenoise.benchmark import calibrate

case = load_released_set(DATA, "PUNet", "sparse", 0.01)
print(f"{case.label}: {len(case.shapes)} shapes")
result = calibrate(case)
print(f"  published Bilateral CD {result['expected_cd']:.2f}")
print(f"  our harness        CD {result['measured_cd']:.2f}  (ratio {result['cd_ratio']:.2f}x)")
print("  PASS" if result["within_tolerance"] else "  FAIL - do not trust the table")
assert result["within_tolerance"], "calibration failed; fix before training"


In [ ]:
from pointdenoise.engine import train

EPOCHS = 60
model, history = train(
    shapes,
    out_dir="/kaggle/working/runs",
    epochs=EPOCHS,
    batch_size=32,
    points_per_patch=256,
    patches_per_shape=1000,
    lr=1e-3,
    repulsion_weight=0.05,
    model_kwargs={"d_model": 256, "num_heads": 8, "num_layers": 6},
    num_workers=2,
    seed=0,
)


In [ ]:
import matplotlib.pyplot as plt

fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4))
a.plot([h["total"] for h in history], label="total")
a.plot([h["chamfer"] for h in history], label="chamfer")
a.set_xlabel("epoch"); a.set_ylabel("loss"); a.legend(); a.grid(alpha=.3)
a.set_title("Training loss")
b.plot([h["lr"] for h in history], color="tab:orange")
b.set_yscale("log"); b.set_xlabel("epoch"); b.set_ylabel("lr"); b.grid(alpha=.3)
b.set_title("Learning rate")
plt.tight_layout(); plt.savefig("/kaggle/working/loss.png", dpi=120); plt.show()

print(f"first epoch {history[0]['total']:.6f} -> last {history[-1]['total']:.6f}")


## Benchmark

Runs the full grid and prints our row into the published comparison table.
`identity` (the noisy input) is scored too, so it is always clear whether the
model helped rather than just how it ranks.


In [ ]:
from pointdenoise.benchmark import NOISE_LEVELS, comparison_table, run_case
from pointdenoise.engine import denoise_cloud

def denoiser(points):
    shape = Shape(np.asarray(points), noisy=np.asarray(points))
    return denoise_cloud(model, shape, points_per_patch=256, batch_size=256, iters=1)

scores, baseline = {}, {}
for resolution in ("sparse", "dense"):
    for noise in NOISE_LEVELS:
        try:
            case = load_released_set(DATA, "PUNet", resolution, noise)
        except FileNotFoundError as exc:
            print("skip", resolution, noise, exc); continue
        _, ours = run_case(case, denoiser, with_p2m=True)
        _, none = run_case(case, lambda p: p, with_p2m=True)
        scores[(resolution, noise)] = ours
        baseline[(resolution, noise)] = none
        print(f"{case.label:<22} ours CD {ours['cd']:7.4f} P2M {ours['p2m']:7.4f}   "
              f"| noisy CD {none['cd']:7.4f}")


In [ ]:
table = comparison_table(scores, our_name="Ours", dataset="PUNet")
print(table)

with open("/kaggle/working/benchmark.txt", "w") as f:
    f.write(table + "\n\nNoisy input baseline\n")
    for k, v in baseline.items():
        f.write(f"  {k[0]}/{k[1]:.0%}  CD {v['cd']:.4f}  P2M {v['p2m']:.4f}\n")
print("\nsaved to /kaggle/working/benchmark.txt")


In [ ]:
import shutil
shutil.copy("/kaggle/working/runs/best.pt", "/kaggle/working/best.pt")
print("download from the output panel: best.pt, benchmark.txt, loss.png")
print(f"best.pt size: {os.path.getsize('/kaggle/working/best.pt')/1e6:.1f} MB")
